In [1]:
import os
import pandas as pd
import numpy as np
import pandas_ta as ta

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)

# =========================
# CONFIGURACIÓN
# =========================

DATA_PATH = "data"
RESULTS_PATH = "results"

os.makedirs(RESULTS_PATH, exist_ok=True)

d = 365
threshold = 0.65

features = [
    "open", "high", "low", "close", "volume",
    "close_smoothed", "rsi", "stochastic_k",
    "williams_r", "macd_line", "macd_signal",
    "macd_hist", "proc", "obv"
]

all_results = []
selected_assets = []

# =========================
# FUNCIÓN DE INDICADORES
# =========================

def prepare_asset_data(df, d=14):
    df = df.copy()
    df.columns = df.columns.str.lower()

    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    diff = df["close"].shift(-d) - df["close"]
    df["target"] = np.where(diff > 0, 1, -1)

    df["close_smoothed"] = df["close"].ewm(alpha=0.1, adjust=False).mean()

    df["rsi"] = ta.rsi(df["close"], length=d)

    df["max"] = df["high"].rolling(window=d).max()
    df["min"] = df["low"].rolling(window=d).min()

    df["stochastic_k"] = 100 * (
        (df["close"] - df["min"]) / (df["max"] - df["min"])
    )

    highest_high = df["high"].rolling(window=d).max()
    lowest_low = df["low"].rolling(window=d).min()

    df["williams_r"] = (
        (highest_high - df["close"]) / (highest_high - lowest_low)
    ) * -100

    price = df["close"]

    ema12 = price.ewm(span=12, adjust=False).mean()
    ema26 = price.ewm(span=26, adjust=False).mean()

    df["macd_line"] = ema12 - ema26
    df["macd_signal"] = df["macd_line"].ewm(span=9, adjust=False).mean()
    df["macd_hist"] = df["macd_line"] - df["macd_signal"]

    df["proc"] = (df["close"] - df["close"].shift(d)) / df["close"].shift(d)

    df["obv"] = 0.0

    for i in range(1, len(df)):
        if df["close"].iloc[i] > df["close"].iloc[i - 1]:
            df.loc[i, "obv"] = df["obv"].iloc[i - 1] + df["volume"].iloc[i]
        elif df["close"].iloc[i] < df["close"].iloc[i - 1]:
            df.loc[i, "obv"] = df["obv"].iloc[i - 1] - df["volume"].iloc[i]
        else:
            df.loc[i, "obv"] = df["obv"].iloc[i - 1]

    df = df.dropna().reset_index(drop=True)

    return df


# =========================
# LOOP POR CADA ASSET
# =========================

for file in os.listdir(DATA_PATH):

    if not file.endswith(".csv"):
        continue

    asset_name = file.replace(".csv", "")
    file_path = os.path.join(DATA_PATH, file)

    print(f"Procesando: {asset_name}")

    df = pd.read_csv(file_path)

    try:
        df = prepare_asset_data(df, d=d)

        if len(df) < 100:
            print(f"Saltado {asset_name}: pocos datos")
            continue

        X = df[features]
        y = df["target"]

        if y.nunique() < 2:
            print(f"Saltado {asset_name}: solo tiene una clase")
            continue

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            shuffle=False
        )

        le = LabelEncoder()
        y_train_enc = le.fit_transform(y_train)
        y_test_enc = le.transform(y_test)

        rf = RandomForestClassifier(
            n_estimators=100,
            max_depth=None,
            min_samples_split=2,
            oob_score=True,
            random_state=42,
            n_jobs=-1
        )

        rf.fit(X_train, y_train_enc)

        y_pred_enc = rf.predict(X_test)
        probs = rf.predict_proba(X_test)

        y_pred = le.inverse_transform(y_pred_enc)
        y_true = y_test.values

        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
        recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0)

        cm = confusion_matrix(y_true, y_pred, labels=[-1, 1])
        tn, fp, fn, tp = cm.ravel()

        specificity = tn / (tn + fp) if (tn + fp) != 0 else 0

        class_index_up = list(le.classes_).index(1)
        prob_up_last = probs[-1, class_index_up]

        try:
            auc = roc_auc_score(y_test_enc, probs[:, class_index_up])
        except:
            auc = np.nan

        result = {
            "asset": asset_name,
            "rows": len(df),
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "specificity": specificity,
            "auc": auc,
            "oob_score": rf.oob_score_,
            "prob_up_last": prob_up_last,
            "last_date": df["date"].iloc[-1],
            "last_close": df["close"].iloc[-1],
            "prediction": "UP" if prob_up_last >= threshold else "NO_UP"
        }

        all_results.append(result)

        if prob_up_last >= threshold:
            selected_assets.append(result)

    except Exception as e:
        print(f"Error en {asset_name}: {e}")


# =========================
# GUARDAR RESULTADOS
# =========================

results_df = pd.DataFrame(all_results)
selected_df = pd.DataFrame(selected_assets)

results_df = results_df.sort_values("prob_up_last", ascending=False)

results_df.to_csv(
    os.path.join(RESULTS_PATH, "random_forest_all_assets_results.csv"),
    index=False
)

selected_df.to_csv(
    os.path.join(RESULTS_PATH, "random_forest_selected_assets_threshold.csv"),
    index=False
)

print("\nRESULTADOS GUARDADOS")
print("Todos los assets:")
print("results/random_forest_all_assets_results.csv")

print("\nAssets filtrados por probabilidad >= 0.75:")
print("results/random_forest_selected_assets_threshold.csv")

print("\nTOP ASSETS")
print(results_df.head(20))

Procesando: 2330.TW
Procesando: 7203.T
Procesando: AAPL
Procesando: AMZN
Procesando: ASML.AS
Procesando: AZN.L
Procesando: BABA
Procesando: BHP
Procesando: BNDX
Procesando: BRK-B
Procesando: BTC-USD
Procesando: BZ=F
Procesando: EMB
Procesando: ETH-USD
Procesando: GC=F
Procesando: HG=F
Procesando: HSBA.L
Procesando: IBE.MC
Procesando: IBGL.L
Procesando: IEF
Procesando: ITX.MC
Procesando: JNJ
Procesando: JNK
Procesando: JPM
Procesando: KC=F
Procesando: LINK-USD
Procesando: MC.PA
Procesando: MSFT
Procesando: MUB
Procesando: NESN.SW
Procesando: NG=F
Procesando: PA=F
Procesando: PG
Procesando: RELIANCE.NS
Procesando: SAN.MC
Procesando: SAP.DE
Procesando: SHY
Procesando: SI=F
Procesando: SIE.DE
Procesando: SOL-USD


c:\Users\jorge\miniconda3\envs\proyecto_finanzas\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Procesando: TCEHY
Procesando: TIP
Procesando: TLT
Procesando: TSLA
Procesando: UNH
Procesando: VALE
Procesando: VTC


c:\Users\jorge\miniconda3\envs\proyecto_finanzas\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Procesando: XOM
Procesando: ZC=F

RESULTADOS GUARDADOS
Todos los assets:
results/random_forest_all_assets_results.csv

Assets filtrados por probabilidad >= 0.75:
results/random_forest_selected_assets_threshold.csv

TOP ASSETS
          asset   rows  accuracy  precision    recall  specificity       auc  \
19          IEF   5613  0.297418   0.226471  1.000000     0.115471  0.640270   
28          MUB   4326  0.434180   0.364721  0.961538     0.174138  0.645497   
33  RELIANCE.NS   7245  0.563837   0.563837  1.000000     0.000000  0.553178   
26        MC.PA   6399  0.503125   0.275194  0.351485     0.573059  0.422803   
29      NESN.SW   8925  0.247059   0.247059  1.000000     0.000000  0.284404   
42          TLT   5613  0.140695   0.054848  1.000000     0.095595  0.545204   
31         PA=F   6115  0.345871   0.220273  1.000000     0.197593  0.630870   
12          EMB   4255  0.561692   0.565321  0.985507     0.005435  0.652914   
23          JPM  11261  0.627164   0.627164  1.000000 